### EpochAI Data Warehouse (`EpochAI_DW`) — Exploratory Data Analysis & Executive Visualization

This notebook connects directly to SQL Server (`EpochAI_DW`), queries the semantic layer (`vw_models_enriched` and `vw_annual_energy_trends`), and builds 7 interactive Plotly visualizations covering FinOps, ESG, and AI governance:

1. **Scaling Laws & The Transparency Gap (2018–2026 YTD):** Tracks median carbon emissions per model alongside compute growth (`log₁₀ FLOP`) and declining industry energy disclosure rates.
2. **Energy Consumption vs. Compute Efficiency (Top 12 Models):** Compares total training energy (`MWh`) against energy intensity (`MWh` per $10^{24}$ operations) to separate cluster scale from hardware efficiency.
3. **Training Budget Breakdown (FinOps):** Quantifies capital allocation between hardware/cluster infrastructure CapEx (~90%) and pure electricity OpEx (~10%), including daily burn rates.
4. **Predictive Scaling Regression & Hidden Demand:** Fits a log-linear scaling model ($R^2$) to project the estimated energy footprint of major undisclosed frontier releases.
5. **Geopolitical Grid Trade-Off & Site Selection:** Contrasts historical national training footprints with a 50,000 MWh *what-if* simulation comparing industrial electricity tariffs against grid carbon intensity.
6. **AI Governance & Reporting Risk:** Examines how the market shift from open-weights to closed commercial API releases drives the environmental reporting gap.
7. **Competitive Landscape (Top 10 AI Labs):** Benchmarks leading organizations by notable model output, peak compute scale, and energy disclosure transparency.

In [1]:
# Install dependencies if needed (run once in terminal):
# pip install pandas sqlalchemy pyodbc plotly nbformat kaleido

import urllib.parse
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import sys; sys.path.append('.'); sys.path.append('..')
from analysis.viz_theme import PALETTE, COLORWAY, apply_editorial_layout, save_chart
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set global Plotly styling
px.defaults.template = "plotly_white"

### 1. Connect to SQL Server (`EpochAI_DW`) & Load Analytical Views

In [2]:
# Update SERVER_NAME if you use a named instance (e.g., r'localhost\SQLEXPRESS')
SERVER_NAME = r"DESKTOP-9IKR9U6\SQLEXPRESS"
DATABASE_NAME = "EpochAI_DW"
DRIVER_NAME = "ODBC Driver 17 for SQL Server"  # Change to 'ODBC Driver 18 for SQL Server' if needed

params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Load semantic views created in 03_views_analise_models.sql
df_master = pd.read_sql("SELECT * FROM vw_models_enriched", engine)
df_annual = pd.read_sql("SELECT * FROM vw_annual_energy_trends ORDER BY publication_year", engine)

print(f"Master View loaded: {df_master.shape[0]:,} models x {df_master.shape[1]} columns")
print(f"Models with valid energy & CO2 data: {df_master['facility_total_energy_mwh'].notna().sum()} ({df_master['facility_total_energy_mwh'].notna().mean()*100:.1f}%)")
df_master.head(3)

Master View loaded: 1,102 models x 45 columns
Models with valid energy & CO2 data: 174 (15.8%)


,model_id,model_name,organization,org_category,publication_date,publication_year,domain,accessibility,open_weights,frontier_model,...,facility_electricity_cost_usd,hardware_infra_cost_usd,GPQA_Diamond_Score,FrontierMath_Score,SWE_Bench_Score,MMLU_Score,WebDev_Arena_Elo,Composite_Capability_Index,usd_per_capability_point,co2_tons_per_capability_point
0,1,Qwen3-VL-235B-A22B,Alibaba,Industry,2025-09-23,2025,"Language,Vision,Multimodal,Video",Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Qwen3-Omni-30B-A3B,Alibaba,Industry,2025-09-22,2025,"Multimodal,Language,Vision,Speech,Video",Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,AgentFounder-30B,Alibaba,Industry,2025-09-16,2025,Language,Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Scaling Laws & The Transparency Gap (2018–2026 YTD)

In [3]:
# Prepare clean annual metrics (2018-2026)
df_trend = (
    df_master[df_master["publication_year"] >= 2018]
    .groupby("publication_year", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("facility_total_energy_mwh", "count"),
        avg_log10_flop=("log10_flop", "mean"),
        median_co2_tons=("pue_adjusted_co2_tons", "median"),
    )
    .sort_values("publication_year")
)

df_trend["transparency_pct"] = (
    (df_trend["disclosed_models"] / df_trend["total_models"]) * 100.0
).round(1)

df_trend["year_label"] = df_trend["publication_year"].apply(
    lambda y: f"{int(y)}*" if int(y) == 2026 else str(int(y))
)

# Executive color palette: Navy Blue (Compute), Coral/Terracotta (Carbon), Slate Teal (Transparency)
NAVY = "#1E3A5F"
CORAL = "#D95F43"
CORAL_LIGHT = "rgba(217, 95, 67, 0.45)"
TEAL = "#2A7B88"
TEAL_FILL = "rgba(42, 123, 136, 0.14)"

bar_colors = [
    CORAL_LIGHT if int(y) == 2026 else CORAL
    for y in df_trend["publication_year"]
]

fig1 = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.18,
    row_heights=[0.64, 0.36],
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
    subplot_titles=(
        "1. Typical Carbon Footprint vs. Computing Power per Model",
        "2. Industry Transparency: Share of Models Disclosing Energy Data (%)",
    ),
)

# Top Panel - Bars: Median CO2 per model
fig1.add_trace(
    go.Bar(
        x=df_trend["year_label"],
        y=df_trend["median_co2_tons"],
        name="Typical Carbon Footprint per Model (Tons CO₂)",
        marker=dict(color=bar_colors, line=dict(color=CORAL, width=1)),
        text=df_trend["median_co2_tons"].apply(
            lambda v: f"{v:,.0f} t" if v >= 10 else f"{v:.1f} t"
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#333333"),
        hovertemplate=(
            "<b>Year: %{x}</b><br>"
            "Typical emissions per model: %{y:,.1f} tons of CO₂<extra></extra>"
        ),
    ),
    row=1,
    col=1,
    secondary_y=False,
)

# Top Panel - Line: Compute Scale
fig1.add_trace(
    go.Scatter(
        x=df_trend["year_label"],
        y=df_trend["avg_log10_flop"],
        name="Average Computing Power (Log Scale)",
        mode="lines+markers",
        line=dict(color=NAVY, width=3),
        marker=dict(size=8, color=NAVY),
        hovertemplate=(
            "<b>Year: %{x}</b><br>"
            "Average computing power: 10<sup>%{y:.1f}</sup> operations<extra></extra>"
        ),
    ),
    row=1,
    col=1,
    secondary_y=True,
)

# Bottom Panel - Area/Line: Transparency Rate (%)
fig1.add_trace(
    go.Scatter(
        x=df_trend["year_label"],
        y=df_trend["transparency_pct"],
        name="Energy Data Disclosure Rate (%)",
        mode="lines+markers+text",
        fill="tozeroy",
        fillcolor=TEAL_FILL,
        line=dict(color=TEAL, width=2.5),
        marker=dict(size=7, color=TEAL),
        text=df_trend["transparency_pct"].apply(lambda p: f"{p:.0f}%"),
        textposition="top center",
        cliponaxis=False,
        textfont=dict(size=11, color=TEAL),
        customdata=np.stack(
            [df_trend["disclosed_models"], df_trend["total_models"]], axis=-1
        ),
        hovertemplate=(
            "<b>Year: %{x}</b><br>"
            "Models with public energy data: %{y:.1f}% (%{customdata[0]} of %{customdata[1]} models)<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig1.update_layout(
    title=dict(
        text=(
            "<b>AI Models Are Getting Bigger and More Carbon-Intensive — Yet Less Transparent (2018–2026)</b><br>"
            "<span style='font-size:13px; color:#555555;'>"
            "While computing power and typical carbon emissions per model surged, public energy disclosure dropped to 5%"
            "</span>"
        ),
        x=0.02,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0.02,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=750,
    margin=dict(t=165, b=80, l=75, r=75),
)

# Clean subtle gridlines
fig1.update_xaxes(showgrid=False, linecolor="#CCCCCC")
fig1.update_yaxes(
    title_text="Typical Emissions per Model (Tons CO₂)",
    range=[0, df_trend["median_co2_tons"].max() * 1.32],
    gridcolor="#EEEEEE",
    row=1,
    col=1,
    secondary_y=False,
)
fig1.update_yaxes(
    title_text="Computing Power (Log₁₀ Scale)",
    range=[17.5, 26.2],
    showgrid=False,
    row=1,
    col=1,
    secondary_y=True,
)
fig1.update_yaxes(
    title_text="Disclosure Rate (%)",
    range=[0, 48],
    gridcolor="#EEEEEE",
    row=2,
    col=1,
)
fig1.update_xaxes(
    title_text="Release Year (*2026 includes data through November — year in progress)",
    row=2,
    col=1,
)

# Position subplot titles cleanly above each panel without overlapping legends or bars
for annotation in fig1["layout"]["annotations"]:
    annotation["font"] = dict(size=13, color="#1E3A5F")
    annotation["x"] = 0.02
    annotation["xanchor"] = "left"
    annotation["yshift"] = 12

fig1.show()
save_chart(fig1, 'chart1_scaling_transparency.png', height=750)


[OK] Saved 4K chart (3600x2250px) -> ..\assets\charts\chart1_scaling_transparency.png


WindowsPath('../assets/charts/chart1_scaling_transparency.png')

### 3. Energy Consumption vs. Compute Efficiency (Top 12 Models)

In [4]:
# Prepare Top 12 models by facility-adjusted energy consumption
df_energy_valid = df_master[
    df_master["facility_total_energy_mwh"].notna() & (df_master["training_compute_flop"] > 0)
].copy()
df_energy_valid["mwh_per_yottaflop"] = df_energy_valid["facility_total_energy_mwh"] / (df_energy_valid["training_compute_flop"] / 1e24)

df_top_energy = (
    df_energy_valid.sort_values("facility_total_energy_mwh", ascending=False)
    .head(12)
    .iloc[::-1]
    .copy()
)

median_intensity = df_top_energy["mwh_per_yottaflop"].median()
intensity_colors = [
    PALETTE["capex_teal"] if v <= median_intensity else PALETTE["carbon_crimson"]
    for v in df_top_energy["mwh_per_yottaflop"]
]

missing_examples = (
    df_master[df_master["facility_total_energy_mwh"].isna() & df_master["log10_flop"].notna()]
    .sort_values("log10_flop", ascending=False)["model_name"]
    .head(3)
    .tolist()
)
missing_note = ", ".join(missing_examples) if missing_examples else "GPT-4, Gemini Ultra"

fig2 = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

fig2.add_trace(
    go.Bar(
        y=df_top_energy["model_name"],
        x=df_top_energy["facility_total_energy_mwh"],
        orientation="h",
        marker_color=PALETTE["disclosed_navy"],
        text=df_top_energy["facility_total_energy_mwh"].apply(lambda v: f"{v:,.0f} MWh"),
        textposition="outside",
        cliponaxis=False,
    ),
    row=1,
    col=1,
)

fig2.add_trace(
    go.Bar(
        y=df_top_energy["model_name"],
        x=df_top_energy["mwh_per_yottaflop"],
        orientation="h",
        marker_color=intensity_colors,
        text=df_top_energy["mwh_per_yottaflop"].apply(lambda v: f"{v:.1f}"),
        textposition="outside",
        cliponaxis=False,
    ),
    row=1,
    col=2,
)
fig2.update_layout(
    title=dict(
        text=(
            "<b>Biggest Energy Consumers vs. Compute Efficiency (Top 12 Disclosed Models)</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            "Newer chips consume more total power due to larger clusters, yet use less energy per unit of compute"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    height=680,
    margin=dict(t=135, b=125, l=160, r=110),
    annotations=[
        dict(
            text="<b>1. Total Energy Consumed (MWh)</b>",
            x=0.0,
            xanchor="left",
            y=1.04,
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#383838"),
        ),
        dict(
            text="<b>2. Energy per 10²⁴ Operations (Lower = Better)</b>",
            x=0.0,
            xanchor="left",
            y=1.04,
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#383838"),
        ),
        dict(
            text=(
                f"<i>Note: Major frontier models without public training-hour disclosures are excluded "
                f"(e.g., {missing_note}).<br>"
                f"Color code (right): <b>Teal</b> = better-than-median efficiency; <b>Coral</b> = higher energy intensity.</i>"
            ),
            x=0.0,
            xanchor="left",
            y=-0.15,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11, color="#666666"),
        ),
    ],
)

fig2.update_xaxes(
    title_text="Total Training Energy (MWh)",
    range=[0, df_top_energy["facility_total_energy_mwh"].max() * 1.32],
    gridcolor="#EEEEEE",
    row=1,
    col=1,
)
fig2.update_xaxes(
    title_text="MWh per 10²⁴ Operations",
    range=[0, df_top_energy["mwh_per_yottaflop"].max() * 1.32],
    gridcolor="#EEEEEE",
    row=1,
    col=2,
)
fig2.update_yaxes(showgrid=False, linecolor="#CCCCCC")

fig2.show()
save_chart(fig2, 'chart2_energy_efficiency.png', height=680)


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart2_energy_efficiency.png


WindowsPath('../assets/charts/chart2_energy_efficiency.png')

### 4. Training Budget Breakdown (FinOps)

In [5]:
# Filter models with valid breakdown (Total Cost > Electricity Cost)
df_valid_cost = df_master.dropna(
    subset=["hardware_infra_cost_usd", "facility_electricity_cost_usd", "total_cost_usd"]
).copy()

df_valid_cost["electricity_share_pct"] = (
    df_valid_cost["facility_electricity_cost_usd"] / df_valid_cost["total_cost_usd"]
) * 100.0

# Industry-wide benchmark across all disclosed budgets
total_valid_count = len(df_valid_cost)
avg_elec_share = df_valid_cost["electricity_share_pct"].mean()

# Top 10 models by total training budget
df_cost = (
    df_valid_cost.sort_values("total_cost_usd", ascending=False)
    .head(10)
    .copy()
)

# Calculate training days and daily burn rate ($/day)
df_cost["training_days"] = (df_cost["training_time_hours"] / 24.0).round(1)
df_cost["daily_burn_usd"] = np.where(
    df_cost["training_days"] > 0,
    df_cost["total_cost_usd"] / df_cost["training_days"],
    np.nan,
)

# Identify top expensive models excluded due to missing electricity/training-hour breakdown
excluded_expensive = (
    df_master[
        df_master["total_cost_usd"].notna()
        & df_master["facility_electricity_cost_usd"].isna()
    ]
    .sort_values("total_cost_usd", ascending=False)
    .head(3)["model_name"]
    .tolist()
)
excluded_note = (
    ", ".join(excluded_expensive)
    if excluded_expensive
    else "GPT-4, Gemini Ultra, Grok 4"
)

NAVY = "#1E3A5F"
CORAL = "#D95F43"

fig3 = go.Figure()

# Bottom bar: Hardware & Cluster Infrastructure CapEx
fig3.add_trace(
    go.Bar(
        name="Hardware & Cluster Infrastructure (CapEx)",
        x=df_cost["model_name"],
        y=df_cost["hardware_infra_cost_usd"],
        marker_color=NAVY,
        customdata=np.stack(
            [
                df_cost["organization"],
                df_cost["chip_family"],
                df_cost["training_days"],
                df_cost["daily_burn_usd"],
                100.0 - df_cost["electricity_share_pct"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>%{x}</b> (%{customdata[0]})<br>"
            "Hardware: %{customdata[1]}<br>"
            "Hardware & Infra Cost: $%{y:,.0f} (%{customdata[4]:.1f}% of budget)<br>"
            "Training Duration: %{customdata[2]:.0f} days (~$%{customdata[3]:,.0f}/day)<extra></extra>"
        ),
    )
)

# Top bar: Pure Electricity Bill OpEx + Total Budget & Power % Label on top
top_labels = [
    f"${tot / 1e6:.1f}M<br><span style='font-size:10px; color:#D95F43;'>({pct:.0f}% power)</span>"
    for tot, pct in zip(df_cost["total_cost_usd"], df_cost["electricity_share_pct"])
]

fig3.add_trace(
    go.Bar(
        name="Electricity Bill (Power OpEx)",
        x=df_cost["model_name"],
        y=df_cost["facility_electricity_cost_usd"],
        marker_color=CORAL,
        text=top_labels,
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#2C3E50"),
        customdata=np.stack(
            [
                df_cost["electricity_share_pct"],
                df_cost["total_cost_usd"],
                df_cost["primary_country"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>%{x}</b> (%{customdata[2]})<br>"
            "Electricity Bill: $%{y:,.0f} (%{customdata[0]:.1f}% of budget)<br>"
            "Total Training Budget: $%{customdata[1]:,.0f}<extra></extra>"
        ),
    )
)

fig3.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "<b>AI Training Cost Structure: Chips Dominate (~90%), Power Is Secondary (~10%)</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            f"Top 10 models with full cost & energy breakdown (across all {total_valid_count} disclosed models, electricity averages {avg_elec_share:.1f}% of total budget)"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=620,
    margin=dict(t=105, b=175, l=80, r=60),
    annotations=[
            dict(
                text=(
                    f"<i>Note: Only {total_valid_count} models in the dataset disclose both total budget and training hours.<br>"
                    f"High-cost models without public energy data are excluded (e.g., {excluded_note}).<br>"
                    f"Top labels show <b>Total Cost ($M)</b> and <b>Electricity Share (%)</b>. Hover for daily burn rate ($/day).</i>"
                ),
                x=0.0,
                xanchor="left",
                y=-0.30,
                yanchor="top",
                xref="paper",
                yref="paper",
                showarrow=False,
                align="left",
                font=dict(size=11, color="#666666"),
            )
        ],
)

fig3.update_xaxes(
    title_text="AI Model (Ranked by Total Disclosed Training Budget)",
    title_standoff=18,
    tickangle=-20,
    showgrid=False,
    linecolor="#CCCCCC",
)
fig3.update_yaxes(
    title_text="Total Training Cost (USD)",
    range=[0, df_cost["total_cost_usd"].max() * 1.25],
    gridcolor="#EEEEEE",
    tickprefix="$",
)

fig3.show()
save_chart(fig3, 'chart3_cost_structure.png', height=680)


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart3_cost_structure.png


WindowsPath('../assets/charts/chart3_cost_structure.png')

### 5. Predictive Scaling Regression & Hidden Demand

In [6]:
# 1. Prepare disclosed models (both compute and energy available)
df_scatter = df_master.dropna(subset=["log10_flop", "facility_total_energy_mwh"]).copy()
df_scatter["training_days"] = (df_scatter["training_time_hours"] / 24.0).round(1)
df_scatter["log10_energy"] = np.log10(df_scatter["facility_total_energy_mwh"])

def group_hardware_gen(chip):
    if chip in ["NVIDIA H100", "NVIDIA H200", "NVIDIA B200"]:
        return "Frontier NVIDIA (H100 / H200 / B200)"
    elif chip == "NVIDIA A100":
        return "NVIDIA A100"
    elif "Google TPU" in str(chip):
        return "Google TPUs (v3 – v5p)"
    else:
        return "Legacy / Other Hardware"

df_scatter["hw_group"] = df_scatter["chip_family"].apply(group_hardware_gen)

# 2. Fit Log-Log Predictive Regression & Confidence Band
slope, intercept = np.polyfit(df_scatter["log10_flop"], df_scatter["log10_energy"], 1)
r_val = np.corrcoef(df_scatter["log10_flop"], df_scatter["log10_energy"])[0, 1]
r_squared = r_val ** 2

# Residual standard error for +/- 1 SD prediction band
y_pred_hist = slope * df_scatter["log10_flop"] + intercept
resid_std = np.std(df_scatter["log10_energy"] - y_pred_hist)

x_line = np.linspace(df_scatter["log10_flop"].min(), 27.3, 100)
y_line_mwh = 10 ** (slope * x_line + intercept)
y_upper_mwh = 10 ** (slope * x_line + intercept + resid_std)
y_lower_mwh = 10 ** (slope * x_line + intercept - resid_std)

# 3. Top 4 Undisclosed Frontier Models (known FLOPs, NULL energy)
df_undisclosed = (
    df_master[df_master["facility_total_energy_mwh"].isna() & df_master["log10_flop"].notna()]
    .sort_values("log10_flop", ascending=False)
    .head(4)
    .copy()
    .reset_index(drop=True)
)
df_undisclosed["predicted_mwh"] = 10 ** (slope * df_undisclosed["log10_flop"] + intercept)
df_undisclosed["rank_num"] = [str(i + 1) for i in range(len(df_undisclosed))]

color_map = {
    "Frontier NVIDIA (H100 / H200 / B200)": "#2A7B88",
    "NVIDIA A100": "#1E3A5F",
    "Google TPUs (v3 – v5p)": "#E69F00",
    "Legacy / Other Hardware": "#95A5A6",
}

fig4 = go.Figure()

# Prediction confidence band (subtle shaded corridor)
fig4.add_trace(
    go.Scatter(
        x=np.concatenate([x_line, x_line[::-1]]),
        y=np.concatenate([y_upper_mwh, y_lower_mwh[::-1]]),
        fill="toself",
        fillcolor="rgba(44, 62, 80, 0.07)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=True,
        name="Typical Estimation Range (±1 SD)",
    )
)

# Disclosed models (muted slightly so projected frontier models stand out clearly)
for group_name, color in color_map.items():
    sub = df_scatter[df_scatter["hw_group"] == group_name]
    if len(sub) == 0:
        continue
    fig4.add_trace(
        go.Scatter(
            x=sub["log10_flop"],
            y=sub["facility_total_energy_mwh"],
            mode="markers",
            name=group_name,
            marker=dict(
                color=color,
                size=np.clip(np.sqrt(sub["pue_adjusted_co2_tons"].fillna(1)) * 0.13 + 6, 6, 30),
                opacity=0.45,
                line=dict(width=0.8, color="white"),
            ),
            customdata=np.stack(
                [
                    sub["model_name"],
                    sub["organization"],
                    sub["chip_family"],
                    sub["training_days"].fillna(0),
                    sub["pue_adjusted_co2_tons"].fillna(0),
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                "Hardware: %{customdata[2]}<br>"
                "Computing Power: 10<sup>%{x:.1f}</sup> operations<br>"
                "Disclosed Energy: %{y:,.0f} MWh (%{customdata[3]:.0f} days)<br>"
                "Carbon Footprint: %{customdata[4]:,.0f} tons CO₂<extra></extra>"
            ),
        )
    )

# Scaling Law Regression Line
fig4.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line_mwh,
        mode="lines",
        name=f"Scaling Law Trendline (R² = {r_squared:.2f})",
        line=dict(color="#1E3A5F", width=2.2, dash="dash"),
        hovertemplate=(
            "<b>Scaling Law Benchmark</b><br>"
            "Computing Power: 10<sup>%{x:.1f}</sup> operations<br>"
            "Expected Energy: %{y:,.0f} MWh<extra></extra>"
        ),
    )
)

# Highlighted Estimated Undisclosed Frontier Models (Numbered Coral Markers)
if len(df_undisclosed) > 0:
    fig4.add_trace(
        go.Scatter(
            x=df_undisclosed["log10_flop"],
            y=df_undisclosed["predicted_mwh"],
            mode="markers+text",
            name="Estimated Undisclosed Models (1–4)",
            marker=dict(
                symbol="circle",
                size=18,
                color="#D95F43",
                line=dict(width=2, color="white"),
            ),
            text=df_undisclosed["rank_num"],
            textposition="middle center",
            textfont=dict(size=10, color="white", family="Segoe UI, Arial, sans-serif"),
            customdata=np.stack(
                [
                    df_undisclosed["model_name"],
                    df_undisclosed["organization"],
                    df_undisclosed["publication_year"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>#%{text} %{customdata[0]}</b> (%{customdata[1]}, %{customdata[2]})<br>"
                "<i>Undisclosed training hours — Model Estimate</i><br>"
                "Computing Power: 10<sup>%{x:.1f}</sup> operations<br>"
                "Estimated Energy: <b>~%{y:,.0f} MWh</b><extra></extra>"
            ),
        )
    )

# Build clean summary card text for the empty bottom-right area
annotations_list = []

# Grok 3 benchmark callout on the left
grok3 = df_scatter[df_scatter["model_name"].str.contains("Grok 3", case=False, na=False)].head(1)
if len(grok3) > 0:
    annotations_list.append(
        dict(
            x=grok3["log10_flop"].values[0],
            y=np.log10(grok3["facility_total_energy_mwh"].values[0]),
            text=f"<b>Benchmark: {grok3['model_name'].values[0]}</b><br>{grok3['facility_total_energy_mwh'].values[0]:,.0f} MWh (Disclosed)",
            showarrow=True,
            arrowhead=2,
            arrowcolor="#1E3A5F",
            ax=-115,
            ay=-15,
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#CCCCCC",
            borderwidth=1,
            borderpad=5,
            font=dict(size=11, color="#1E3A5F"),
        )
    )

# Clean Executive Summary Box in the empty bottom-right quadrant listing Models 1-4
if len(df_undisclosed) > 0:
    lines = ["<b>Estimated Energy — Top Undisclosed Models</b>"]
    for _, r in df_undisclosed.iterrows():
        lines.append(
            f"<b>#{r['rank_num']} {r['model_name']}</b> ({r['organization']}): <b>~{r['predicted_mwh']:,.0f} MWh</b>"
        )
    box_text = "<br>".join(lines)

    annotations_list.append(
        dict(
            text=box_text,
            x=0.98,
            xanchor="right",
            y=0.06,
            yanchor="bottom",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            bgcolor="rgba(255, 248, 246, 0.96)",
            bordercolor="#D95F43",
            borderwidth=1.5,
            borderpad=10,
            font=dict(size=11, color="#2C3E50"),
        )
    )

# Bottom methodology footnote
annotations_list.append(
    dict(
        text=(
            "<i>Note: Bubble size reflects CO₂ emissions. Numbered coral circles (1–4) project energy demand for major closed models "
            "from their disclosed compute scale.<br>"
            "Models positioned <b>below</b> the dashed trendline achieved higher energy efficiency than the historical average.</i>"
        ),
        x=0.0,
        xanchor="left",
        y=-0.20,
        yanchor="top",
        xref="paper",
        yref="paper",
        showarrow=False,
        align="left",
        font=dict(size=11, color="#666666"),
    )
)

fig4.update_layout(
    title=dict(
        text=(
            f"<b>Estimating Hidden AI Energy Demand: Compute Explains {r_squared * 100:.0f}% of Energy Growth</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            "Log-scale regression across disclosed models enables reliable energy estimates for top undisclosed frontier releases"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=700,
    margin=dict(t=165, b=135, l=80, r=60),
    annotations=annotations_list,
)

fig4.update_xaxes(
    title_text="Training Computing Power (Log₁₀ Operations — Each +1 = 10x More Compute)",
    title_standoff=14,
    range=[13.5, 28.0],
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
)
fig4.update_yaxes(
    title_text="Total Energy Consumed (MWh — Log Scale)",
    type="log",
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
)

fig4.show()
save_chart(fig4, 'chart4_predictive_scaling.png', height=680)


C:\Users\marce\OneDrive\Documentos\SQL Projects\epoch-ai-models-eda\.venv-1\Lib\site-packages\pandas\core\arraylike.py:438: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\marce\OneDrive\Documentos\SQL Projects\epoch-ai-models-eda\.venv-1\Lib\site-packages\numpy\lib\_function_base_impl.py:2896: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
C:\Users\marce\OneDrive\Documentos\SQL Projects\epoch-ai-models-eda\.venv-1\Lib\site-packages\numpy\lib\_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in dot
  c = dot(X, X_T.conj())


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart4_predictive_scaling.png


WindowsPath('../assets/charts/chart4_predictive_scaling.png')

### 6. Geopolitical Grid Trade-Off & Site Selection

In [7]:
# Prepare historical country footprint + 50,000 MWh site-selection simulation
df_country_hist = (
    df_master[
        df_master["facility_total_energy_mwh"].notna()
        & df_master["primary_country"].notna()
        & (~df_master["primary_country"].isin(["Unknown", "Multinational", ""]))
    ]
    .groupby("primary_country", as_index=False)
    .agg(
        hist_mwh=("facility_total_energy_mwh", "sum"),
        hist_co2=("pue_adjusted_co2_tons", "sum"),
        tariff_usd_mwh=("tariff_usd_mwh", "median"),
        carbon_gco2_kwh=("carbon_gco2_kwh", "median"),
    )
    .sort_values("hist_mwh", ascending=True)
)

df_country = df_country_hist.dropna(subset=["tariff_usd_mwh", "carbon_gco2_kwh"]).copy()
df_country["sim_cost_usd_m"] = (50000.0 * df_country["tariff_usd_mwh"]) / 1e6
df_country["sim_co2_tons"] = (50000.0 * 1000.0 * df_country["carbon_gco2_kwh"]) / 1e6

fig5 = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.14,
)

# Panel 1: Historical Energy vs CO2 (Grouped Horizontal Bars)
fig5.add_trace(
    go.Bar(
        y=df_country["primary_country"],
        x=df_country["hist_mwh"],
        name="Historical Energy (MWh)",
        orientation="h",
        marker_color=PALETTE["disclosed_navy"],
        text=df_country["hist_mwh"].apply(lambda v: f"{v:,.0f}"),
        textposition="outside",
        cliponaxis=False,
    ),
    row=1,
    col=1,
)

fig5.add_trace(
    go.Bar(
        y=df_country["primary_country"],
        x=df_country["hist_co2"],
        name="Historical Carbon (Tons CO₂)",
        orientation="h",
        marker_color=PALETTE["carbon_crimson"],
        text=df_country["hist_co2"].apply(lambda v: f"{v:,.0f}"),
        textposition="outside",
        cliponaxis=False,
    ),
    row=1,
    col=1,
)

# Panel 2: 50,000 MWh What-If Quadrant Scatter
fig5.add_trace(
    go.Scatter(
        x=df_country["sim_cost_usd_m"],
        y=df_country["sim_co2_tons"],
        mode="markers+text",
        text=df_country["primary_country"],
        textposition="top center",
        marker=dict(size=14, color=PALETTE["capex_teal"], line=dict(width=1.5, color="white")),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig5.add_vline(x=df_country["sim_cost_usd_m"].median(), line_dash="dot", line_color="#94A3B8", row=1, col=2)
fig5.add_hline(y=df_country["sim_co2_tons"].median(), line_dash="dot", line_color="#94A3B8", row=1, col=2)
fig5.update_layout(
    barmode="group",
    title=dict(
        text=(
            "<b>Geopolitical Grid Trade-Off: Where Is the Best Hub to Train Frontier AI?</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            "The US accounts for >90% of disclosed training energy, while Canada offers the best balance of low cost and clean power"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.97,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.10,        
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=680,
    margin=dict(t=135, b=150, l=125, r=65), 
    annotations=[
        dict(
            text="<b>1. Historical Footprint (MWh vs. Tons CO₂)</b>",
            x=0.0,
            xanchor="left",
            y=1.02,
            yanchor="bottom",
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#666666"),
        ),
        dict(
            text="<b>2. What-If Simulation: 50,000 MWh Run</b>",
            x=0.0,
            xanchor="left",
            y=1.02,
            yanchor="bottom",
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#666666"),
        ),
        # Highlight Sweet Spot (Bottom-Left Quadrant)
        dict(
            text="<b>Optimal Quadrant</b><br>(Low Cost & Low Carbon)",
            x=df_country["sim_cost_usd_m"].min() * 0.80,
            y=df_country["sim_co2_tons"].max() * 0.10,
            xref="x2",
            yref="y2",
            showarrow=False,
            align="left",
            font=dict(size=10, color="#666666"),
            bgcolor="rgba(42, 123, 136, 0.08)",
            borderpad=4,
        ),
        dict(
            text=(
                "<i>Note: Left panel shows models with public energy data (excluding unassigned/multinational runs).<br>"
                "Right panel simulates training a 50,000 MWh cluster using national industrial tariffs and grid carbon intensity.<br>"
                "Dotted lines indicate median cost and carbon thresholds across analyzed hubs.</i>"
            ),
            x=0.0,
            xanchor="left",
            y=-0.18,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11, color="#666666"),
        ),
    ],
)

# Left panel axes (Log scale on X with generous headroom so outside labels never touch Panel 2)
fig5.update_xaxes(
    title_text="Volume (Log Scale: MWh & Tons CO₂)",
    title_standoff=14,
    type="log",
    range=[1.5, 7.2],
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row=1,
    col=1,
)
fig5.update_yaxes(showgrid=False, linecolor="#CCCCCC", row=1, col=1)

# Right panel axes (Linear Cost vs Linear Carbon)
fig5.update_xaxes(
    title_text="Simulated Electricity Bill ($ Millions USD)",
    title_standoff=14,
    range=[df_country["sim_cost_usd_m"].min() * 0.80, df_country["sim_cost_usd_m"].max() * 1.18],
    tickprefix="$",
    ticksuffix="M",
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row=1,
    col=2,
)
fig5.update_yaxes(
    title_text="Simulated Emissions (Tons CO₂)",
    range=[0, df_country["sim_co2_tons"].max() * 1.18],
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row=1,
    col=2,
)

fig5.show()
save_chart(fig5, 'chart5_geopolitical_grid.png', height=680)


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart5_geopolitical_grid.png


WindowsPath('../assets/charts/chart5_geopolitical_grid.png')

### 7. AI Governance & Reporting Risk

In [8]:
# Prepare AI Governance & Accessibility segmentation safely from df_master
df_gov = df_master[df_master["publication_year"] >= 2018].copy()

# Detect accessibility column if present in vw_models_enriched, otherwise classify by known open-weights releases
access_col = None
for candidate in ["model_accessibility", "accessibility", "Model accessibility"]:
    if candidate in df_gov.columns:
        access_col = candidate
        break

def classify_governance(row):
    if access_col and pd.notna(row[access_col]):
        val = str(row[access_col]).lower()
        if "open" in val:
            return "Open-Weights / Public"
        elif "api" in val or "hosted" in val or "closed" in val or "unreleased" in val:
            return "Closed / Commercial API"
    name_org = f"{row.get('model_name', '')} {row.get('organization', '')}".lower()
    open_keywords = [
        "llama", "mistral", "mixtral", "deepseek", "qwen", "gemma",
        "falcon", "olmo", "pythia", "bloom", "opt-", "t5", "bert", "yi-", "phi-"
    ]
    if any(k in name_org for k in open_keywords):
        return "Open-Weights / Public"
    return "Closed / Commercial API"

df_gov["release_model"] = df_gov.apply(classify_governance, axis=1)
df_gov["has_energy"] = df_gov["facility_total_energy_mwh"].notna()

# Left Panel Data: Annual Share (%) of Open-Weights vs Closed Models (2018-2026)
df_gov_year = (
    df_gov.groupby(["publication_year", "release_model"], as_index=False)
    .agg(models_count=("model_id", "count"))
)
df_year_tot = df_gov.groupby("publication_year", as_index=False).agg(year_total=("model_id", "count"))
df_gov_year = df_gov_year.merge(df_year_tot, on="publication_year")
df_gov_year["share_pct"] = (df_gov_year["models_count"] / df_gov_year["year_total"]) * 100.0
df_gov_year["year_label"] = df_gov_year["publication_year"].apply(
    lambda y: f"{int(y)}*" if int(y) == 2026 else str(int(y))
)

# Right Panel Data: Energy Disclosure Rate (%) & Median Compute by Release Model
df_gov_summary = (
    df_gov.groupby("release_model", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("has_energy", "sum"),
        median_log10_flop=("log10_flop", "median"),
    )
)
df_gov_summary["disclosure_pct"] = (
    df_gov_summary["disclosed_models"] / df_gov_summary["total_models"]
) * 100.0

NAVY = "#1E3A5F"
TEAL = "#2A7B88"
CORAL = "#D95F43"

fig6 = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.14,
    column_widths=[0.56, 0.44],
    subplot_titles=(
        "1. Annual Share of Model Releases (2018–2026*)",
        "2. Energy Data Disclosure Rate (%)",
    ),
)

# Left Panel: 100% Stacked Bars over time
for rel_type, color in [
    ("Open-Weights / Public", TEAL),
    ("Closed / Commercial API", NAVY),
]:
    sub = df_gov_year[df_gov_year["release_model"] == rel_type].sort_values("publication_year")
    fig6.add_trace(
        go.Bar(
            x=sub["year_label"],
            y=sub["share_pct"],
            name=rel_type,
            marker_color=color,
            text=sub["share_pct"].apply(lambda p: f"{p:.0f}%" if p >= 12 else ""),
            textposition="inside",
            insidetextanchor="middle",
            textfont=dict(size=10, color="white"),
            customdata=np.stack([sub["models_count"], sub["year_total"]], axis=-1),
            hovertemplate=(
                "<b>%{x} — " + rel_type + "</b><br>"
                "Share: %{y:.1f}% (%{customdata[0]} of %{customdata[1]} models)<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Right Panel: Disclosure Rate (%) comparison
bar_colors_r = [
    TEAL if "Open" in m else CORAL for m in df_gov_summary["release_model"]
]
fig6.add_trace(
    go.Bar(
        x=df_gov_summary["release_model"],
        y=df_gov_summary["disclosure_pct"],
        name="Energy Transparency Rate (%)",
        showlegend=False,
        marker_color=bar_colors_r,
        width=0.48,
        text=df_gov_summary.apply(
            lambda r: f"{r['disclosure_pct']:.1f}%<br>({int(r['disclosed_models'])}/{int(r['total_models'])})",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#2C3E50"),
        customdata=df_gov_summary["median_log10_flop"],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Energy Disclosure Rate: %{y:.1f}%<br>"
            "Median Computing Power: 10<sup>%{customdata:.1f}</sup> operations<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig6.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "<b>AI Governance & Risk: Commercial Closure Drives the Energy Transparency Gap</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            "As frontier AI shifted toward closed commercial releases, public reporting of training energy dropped sharply"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.97,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=640,
    margin=dict(t=155, b=145, l=70, r=60),
    annotations=[
        dict(
            text="<b>1. Annual Share of Model Releases (2018–2026*)</b>",
            x=0.0,
            xanchor="left",
            y=1.02,
            yanchor="bottom",
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color=NAVY),
        ),
        dict(
            text="<b>2. Energy Data Disclosure Rate (%)</b>",
            x=0.0,
            xanchor="left",
            y=1.02,
            yanchor="bottom",
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color=TEAL),
        ),
        dict(
            text=(
                "<i>Note: Covers all notable AI models released since 2018 (*2026 data through November).<br>"
                "Labels on the right panel show disclosed models vs. total models in each release category.</i>"
            ),
            x=0.0,
            xanchor="left",
            y=-0.22,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11, color="#666666"),
        ),
    ],
)

fig6.update_xaxes(title_text="Release Year (*2026 YTD)", showgrid=False, linecolor="#CCCCCC", row=1, col=1)
fig6.update_yaxes(title_text="Share of Annual Releases (%)", range=[0, 105], gridcolor="#EEEEEE", row=1, col=1)

fig6.update_xaxes(title_text="Release Strategy", showgrid=False, linecolor="#CCCCCC", row=1, col=2)
fig6.update_yaxes(
    title_text="Models Disclosing Energy (%)",
    range=[0, df_gov_summary["disclosure_pct"].max() * 1.38],
    gridcolor="#EEEEEE",
    row=1,
    col=2,
)

fig6.show()
save_chart(fig6, 'chart6_governance_risk.png', height=680)


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart6_governance_risk.png


WindowsPath('../assets/charts/chart6_governance_risk.png')

### 8. Competitive Landscape (Top 10 AI Labs)

In [9]:
# Top 10 AI Organizations Leaderboard: Model Output, Peak Compute Scale & Energy Transparency
df_orgs = (
    df_master.dropna(subset=["organization"])
    .assign(organization=lambda d: d["organization"].astype(str).str.split(",").str[0].str.strip())
    .query("organization != '' and organization != 'None' and organization != 'nan'")
    .groupby("organization", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("facility_total_energy_mwh", "count"),
        max_log10_flop=("log10_flop", "max"),
        total_disclosed_mwh=("facility_total_energy_mwh", "sum"),
    )
    .sort_values(["total_models", "max_log10_flop"], ascending=[False, False])
    .head(10)
    .sort_values("total_models", ascending=True)
    .copy()
)

df_orgs["transparency_pct"] = (
    (df_orgs["disclosed_models"] / df_orgs["total_models"]) * 100.0
).round(1)

top10_share_pct = (df_orgs["total_models"].sum() / len(df_master)) * 100.0

NAVY = "#1E3A5F"
TEAL = "#2A7B88"
CORAL = "#D95F43"

fig7 = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    column_widths=[0.54, 0.46],
    subplot_titles=(
        "1. Model Volume & Peak Compute Scale",
        "2. Energy Disclosure Rate (%)",
    ),
)

# Left Panel: Total Models Released + Peak Compute Scale
fig7.add_trace(
    go.Bar(
        y=df_orgs["organization"],
        x=df_orgs["total_models"],
        orientation="h",
        name="Notable Models Released",
        marker_color=NAVY,
        text=df_orgs.apply(
            lambda r: f"{int(r['total_models'])} models (Peak: 10^{r['max_log10_flop']:.1f})"
            if pd.notna(r["max_log10_flop"])
            else f"{int(r['total_models'])} models",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=10, color="#2C3E50"),
        customdata=df_orgs["total_disclosed_mwh"],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Total Notable Models: %{x}<br>"
            "Disclosed Training Energy: %{customdata:,.0f} MWh<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# Right Panel: Transparency Rate (%) color-coded (Teal >= 15%, Coral < 15%)
transp_colors = [
    TEAL if p >= 15.0 else CORAL for p in df_orgs["transparency_pct"]
]

fig7.add_trace(
    go.Bar(
        y=df_orgs["organization"],
        x=df_orgs["transparency_pct"],
        orientation="h",
        name="Energy Transparency (%)",
        marker_color=transp_colors,
        text=df_orgs.apply(
            lambda r: f"{r['transparency_pct']:.0f}% ({int(r['disclosed_models'])}/{int(r['total_models'])})",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=10, color="#2C3E50"),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Energy Transparency: %{x:.1f}%<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig7.update_layout(
    title=dict(
        text=(
            "<b>Competitive Landscape: Top 10 AI Labs Drive "
            f"{top10_share_pct:.0f}% of All Notable Models</b><br>"
            "<span style='font-size:12px; color:#555555;'>"
            "Comparing model output, peak computing scale, and environmental reporting transparency across leading labs"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.97,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    height=650,
    margin=dict(t=125, b=135, l=150, r=95),
    annotations=[
        dict(
            text="<b>1. Model Volume & Peak Compute Scale</b>",
            x=0.0,
            xanchor="left",
            y=1.03,
            yanchor="bottom",
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color=NAVY),
        ),
        dict(
            text="<b>2. Energy Disclosure Rate (%)</b>",
            x=0.0,
            xanchor="left",
            y=1.03,
            yanchor="bottom",
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color=TEAL),
        ),
        dict(
            text=(
                "<i>Note: Organizations are attributed by lead developer. Peak scale shows highest disclosed training compute (log₁₀ FLOP).<br>"
                "Right panel color code: <b>Teal</b> = ≥15% energy disclosure rate; <b>Coral</b> = <15% disclosure rate.</i>"
            ),
            x=0.0,
            xanchor="left",
            y=-0.20,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11, color="#666666"),
        ),
    ],
)

fig7.update_xaxes(
    title_text="Number of Notable Models",
    title_standoff=14,
    range=[0, df_orgs["total_models"].max() * 1.48],
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row=1,
    col=1,
)
fig7.update_xaxes(
    title_text="Energy Disclosure Rate (%)",
    title_standoff=14,
    range=[0, max(df_orgs["transparency_pct"].max() * 1.35, 35)],
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row=1,
    col=2,
)
fig7.update_yaxes(showgrid=False, linecolor="#CCCCCC")

fig7.show()
save_chart(fig7, 'chart7_competitive_landscape.png', height=680)


[OK] Saved 4K chart (3600x2040px) -> ..\assets\charts\chart7_competitive_landscape.png


WindowsPath('../assets/charts/chart7_competitive_landscape.png')